In [1]:
import os
import warnings
import logging
%pip install -q sktime scikit-optimize sklearn-genetic-opt
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_VLOG_LEVEL'] = '3'
os.environ['ABSL_CPP_MIN_LOG_LEVEL'] = '3'

warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('absl').setLevel(logging.ERROR)

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils.validation import check_is_fitted
import sys

sys.path.append('/kaggle/input/datasets/keithmarange/lgbm-util/')
sys.path.append('/kaggle/input/cmi-competition-code')
import utils

import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(3)

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import ParameterSampler, RandomizedSearchCV
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
from sklearn_genetic import GASearchCV
from sklearn_genetic.space import Continuous, Categorical as GenCategorical, Integer as GenInteger
import importlib

from scipy.stats import randint, uniform, loguniform

from lightgbm import LGBMClassifier

from scipy.stats import randint, uniform, loguniform
from skopt.space import Categorical, Integer, Real
from sklearn_genetic.space import Categorical as GenCategorical
from sklearn_genetic.space import Integer as GenInteger
from sklearn_genetic.space import Continuous

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

tf.keras.backend.clear_session()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 8.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


E0000 00:00:1777462796.202179      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777462796.278835      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777462796.933556      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777462796.933593      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777462796.933595      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777462796.933597      16 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


In [3]:
feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_w', 'rot_x', 'rot_y', 'rot_z']
thm_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
sampling_rate = 100

pipe_name = 'imu_extractor'
model_run_name = 'lgbm_v2.pkl'
path_to_model_run_name = model_run_folder_name + model_run_name

n_splits = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]
tof_1 = [f'tof_1_v{j}' for j in range(64)]

linear_edges = np.arange(0, 51, 10)
log_edges    = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

model_target = 'gesture_action'
search_mode  = 'grid'  # Options: 'grid', 'random', 'bayesian', 'evolutionary'

candidates = 10
generations = 5
tournament_size = 3
elitism = True

train_size = 0.2

In [4]:
train_df = raw_train_df.set_index('row_id')

if train_size is None:
    rows = (train_demo_df['adult_child'] == 1) & (train_demo_df['sex'] == 1) & (train_demo_df['handedness'] == 1)
    ideal_subject_ids = train_demo_df.loc[rows].sort_values(by='elbow_to_wrist_cm', ascending=False)['subject'].to_list()

    train_sample_df = train_df.loc[train_df['subject'].isin(ideal_subject_ids), :]
    train_sample_df = train_sample_df[train_sample_df['sequence_type'] == 'Target']
    train_sample_df['gesture_action'] = train_sample_df['gesture'].str.split(' - ').str[-1]
    print(train_sample_df['sequence_id'].nunique())
else:
    target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

    target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
    target_only_df['gesture_action']   = target_only_df['gesture'].str.split(' - ').str[-1]
    target_only_df = target_only_df[target_only_df['phase'] == 'Gesture']

    train_sample_df, test_sample_df = utils.sample_balanced_split(
        target_only_df,
        train_pct=train_size,
        test_pct=0.2
    )

# some_sequences = train_sample_df['sequence_id'].unique()[50:]
# train_sample_df = train_sample_df[train_sample_df['sequence_id'].isin(some_sequences)]

if model_target == 'gesture_action':
    train_sample_df.drop(columns=['gesture_position', 'gesture'], inplace=True)
elif model_target == 'gesture':
    train_sample_df.drop(columns=['gesture_position', 'gesture_action'], inplace=True)
else:
    train_sample_df.drop(columns=['gesture', 'gesture_action'], inplace=True)


Train: 648 seqs | 12.7%
Test:  648 seqs  | 12.7%


In [5]:
pipe_name = 'imu_extractor'
clf_prefix = 'classifier__estimator__'

num_pattern  = 'acc|rot|thm|tof'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
cat_cols     = ['orientation']
normal_cols  = ['adult_child', 'sex', 'handedness']
ordinal_cols = ['segment_id']

preprocessor = ColumnTransformer(
    transformers=[
        ('feature_num_cols', StandardScaler(), make_column_selector(pattern=num_pattern)),
        ('subject_num_cols', StandardScaler(), utils.existing_cols(suspect_cols)),
        ('cat_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), utils.existing_cols(cat_cols)),
        ('normal_cols', 'passthrough', utils.existing_cols(normal_cols)),
        ('ordinal_cols', 'passthrough', utils.existing_cols(ordinal_cols))
    ],
    remainder='drop',
    verbose_feature_names_out=False
)
preprocessor.set_output(transform='pandas')

# Fixed values (not tuned)
fixed_params = {
    f'{pipe_name}__imu_sensor_list':        [acc_columns],
    f'{pipe_name}__rotation_sensor_list':   [None],
    f'{pipe_name}__sampling_rate':          [100],
    f'{pipe_name}__subject_df':             [train_demo_df],          # not tuned
    f'{pipe_name}__disable_tqdm':           [True],
    f'{pipe_name}__tof_sensor_list':        [None],                   # unused but present
    f'{pipe_name}__band_edges':             [linear_edges],                   # can also be tuned (see below)
    f'{pipe_name}__dc_offset':              [0],
}

if search_mode == 'grid':
    param_grid = {
        **fixed_params,

        # ----- ImuExtractor -----
        f'{pipe_name}__imu_domain':             ['time'],           # FFT or time features
        f'{pipe_name}__rotation_domain':        ['orientation'],          # quaternion‑based or differences
        f'{pipe_name}__thermopile_sensor_list': [thm_columns],      # single or two sensors
        f'{pipe_name}__thermopile_mode':        ['baseline'],
        f'{pipe_name}__tof_mode':               ['baseline'],     # ToF feature extraction mode
        f'{pipe_name}__category_data':          [True],                       # include subject/orientation
        f'{pipe_name}__segmentation':           [None],           # grouping method
        f'{pipe_name}__window':                 [0.5],                    # seconds (only if segmentation='window')
        f'{pipe_name}__step_sec':               [0.2],                    # step (only if window)
        f'{pipe_name}__combine_imu_axes':       [True],
        f'{pipe_name}__combine_rot_axes':       [True],

        # ----- LGBMClassifier -----
        f'{clf_prefix}n_estimators':            [100],
        f'{clf_prefix}max_depth':               [3],
        f'{clf_prefix}learning_rate':           [0.01],
        f'{clf_prefix}subsample':               [0.6],
        f'{clf_prefix}colsample_bytree':        [0.6],
        f'{clf_prefix}min_child_samples':       [10],
        f'{clf_prefix}reg_alpha':               [0.01],
        f'{clf_prefix}reg_lambda':              [0.01],
        f'{clf_prefix}num_leaves':              [15],
        f'{clf_prefix}min_gain_to_split':       [0.0],
    }

elif search_mode == 'random':
    param_grid = {
        **fixed_params,

        f'{pipe_name}__imu_domain':             ['time', 'acceleration', 'velocity', 'displacement'],
        f'{pipe_name}__rotation_domain':        ['orientation', 'motion', 'frequency'],
        f'{pipe_name}__thermopile_sensor_list': [['thm_1'], ['thm_1','thm_2'], ['thm_1','thm_3','thm_5']],
        f'{pipe_name}__thermopile_mode':        ['baseline', 'spatial'],
        f'{pipe_name}__tof_mode':               ['baseline', 'spatial', 'edge', 'fft2', 'svd', 'wavelet'],
        f'{pipe_name}__category_data':          [True, False],
        f'{pipe_name}__segmentation':           [None, 'window', 'phase', 'behavior', 'both'],
        f'{pipe_name}__window':                 uniform(0.2, 2.0),          # 0.2 to 2.2
        f'{pipe_name}__step_sec':               uniform(0.05, 1.0),         # 0.05 to 1.05
        f'{pipe_name}__combine_imu_axes':       [True, False],
        f'{pipe_name}__combine_rot_axes':       [True, False],

        f'{clf_prefix}n_estimators':            randint(50, 1000),
        f'{clf_prefix}max_depth':               randint(2, 15),
        f'{clf_prefix}learning_rate':           loguniform(0.005, 0.5),
        f'{clf_prefix}subsample':               uniform(0.5, 0.5),           # 0.5–1.0
        f'{clf_prefix}colsample_bytree':        uniform(0.5, 0.5),
        f'{clf_prefix}min_child_samples':       randint(5, 200),
        f'{clf_prefix}reg_alpha':               loguniform(1e-3, 10.0),
        f'{clf_prefix}reg_lambda':              loguniform(1e-3, 10.0),
        f'{clf_prefix}num_leaves':              randint(4, 128),
        f'{clf_prefix}min_gain_to_split':       uniform(0.0, 1.0),
    }

elif search_mode == 'bayesian':
    param_grid = {
        # Choices: 0 = None, 1 = use sensor
        f'{pipe_name}__imu_choice':          Categorical([0, 1]),
        f'{pipe_name}__rot_choice':          Categorical([0, 1]),
        f'{pipe_name}__tof_choice':          Categorical([0, 1]),
        f'{pipe_name}__thermo_choice':       Categorical([0, 1]),
        f'{pipe_name}__band_edges_choice':   Categorical([0, 1, 2, 3]),

        f'{pipe_name}__imu_domain':          Categorical(['time', 'acceleration', 'velocity', 'displacement']),
        f'{pipe_name}__rotation_domain':     Categorical(['orientation', 'motion', 'frequency']),
        f'{pipe_name}__thermopile_mode':     Categorical(['baseline', 'spatial']),
        f'{pipe_name}__tof_mode':            Categorical(['baseline', 'spatial', 'edge', 'fft2', 'svd', 'wavelet']),
        f'{pipe_name}__category_data':       Categorical([False]),
        f'{pipe_name}__segmentation':        Categorical(['window']),
        f'{pipe_name}__window':              Real(0.2, 2.0, prior='uniform'),
        f'{pipe_name}__step_sec':            Real(0.05, 1.0, prior='uniform'),
        f'{pipe_name}__combine_imu_axes':    Categorical([True, False]),
        f'{pipe_name}__combine_rot_axes':    Categorical([True, False]),

        f'{clf_prefix}n_estimators':         Integer(50, 1000),
        f'{clf_prefix}max_depth':            Integer(2, 15),
        f'{clf_prefix}learning_rate':        Real(0.005, 0.5, prior='log-uniform'),
        f'{clf_prefix}subsample':            Real(0.5, 1.0),
        f'{clf_prefix}colsample_bytree':     Real(0.5, 1.0),
        f'{clf_prefix}min_child_samples':    Integer(5, 200),
        f'{clf_prefix}reg_alpha':            Real(1e-3, 10.0, prior='log-uniform'),
        f'{clf_prefix}reg_lambda':           Real(1e-3, 10.0, prior='log-uniform'),
        f'{clf_prefix}num_leaves':           Integer(4, 128),
        f'{clf_prefix}min_gain_to_split':    Real(0.0, 1.0),
    }

elif search_mode == 'evolutionary':
    param_grid = {
        # Integer choices (0=None, 1=use)
        f'{pipe_name}__imu_choice':          GenCategorical([0, 1]),
        f'{pipe_name}__rot_choice':          GenCategorical([0, 1]),
        f'{pipe_name}__tof_choice':          GenCategorical([0, 1]),
        f'{pipe_name}__thermo_choice':       GenCategorical([0, 1]),
        f'{pipe_name}__band_edges_choice':   GenCategorical([0, 1, 2, 3]),

        f'{pipe_name}__imu_domain':          GenCategorical(['time', 'acceleration', 'velocity', 'displacement']),
        f'{pipe_name}__rotation_domain':     GenCategorical(['orientation', 'motion', 'frequency']),
        f'{pipe_name}__thermopile_mode':     GenCategorical(['baseline', 'spatial']),
        f'{pipe_name}__tof_mode':            GenCategorical(['baseline', 'spatial', 'edge', 'fft2', 'svd', 'wavelet']),
        f'{pipe_name}__category_data':       GenCategorical([False]),
        f'{pipe_name}__segmentation':        GenCategorical(['window']),
        f'{pipe_name}__window':              Continuous(0.1, 2.0),
        f'{pipe_name}__step_sec':            Continuous(0.01, 1.0),
        f'{pipe_name}__combine_imu_axes':    GenCategorical([True, False]),
        f'{pipe_name}__combine_rot_axes':    GenCategorical([True, False]),

        f'{clf_prefix}n_estimators':         GenInteger(50, 1000),
        f'{clf_prefix}max_depth':            GenInteger(2, 15),
        f'{clf_prefix}learning_rate':        Continuous(0.005, 0.5),
        f'{clf_prefix}subsample':            Continuous(0.5, 1.0),
        f'{clf_prefix}colsample_bytree':     Continuous(0.5, 1.0),
        f'{clf_prefix}min_child_samples':    GenInteger(5, 200),
        f'{clf_prefix}reg_alpha':            Continuous(1e-3, 10.0),
        f'{clf_prefix}reg_lambda':           Continuous(1e-3, 10.0),
        f'{clf_prefix}num_leaves':           GenInteger(4, 128),
        f'{clf_prefix}min_gain_to_split':    Continuous(0.0, 1.0),
    }

In [6]:
if search_mode in ['evolutionary', 'bayesian']:
    custom_extractor = utils.ImuExtractorWithChoice(subject_df=train_demo_df, dc_offset=0)
else:
    custom_extractor = utils.ImuExtractor(subject_df=train_demo_df)
    
lgbm_clf = LGBMClassifier(objective='multiclass', random_state=42, verbose=-1)

pipeline = Pipeline([
    (pipe_name, custom_extractor),
    ('preprocessor', preprocessor),
    ('classifier', utils.ManyToOneWrapper(
        estimator=lgbm_clf,
        extractor=custom_extractor,
        mode=None,
        target=model_target
    ))
])

cv = GroupKFold(n_splits=n_splits)

if search_mode == 'grid':
    print('Grid Search')
    search_obj = GridSearchCV(
        pipeline, param_grid, cv=cv, verbose=2, n_jobs=-1, return_train_score=True
    )
elif search_mode == 'random':
    print('Random Search')
    search_obj = RandomizedSearchCV(
        pipeline, param_grid, n_iter=candidates, cv=cv, verbose=2, n_jobs=-1, random_state=42, return_train_score=True
    )
elif search_mode == 'bayesian':
    print('Bayesian Search')
    search_obj = BayesSearchCV(
        pipeline, param_grid, n_iter=candidates, cv=cv, verbose=3, n_jobs=-1, random_state=42, return_train_score=True
    )
elif search_mode == 'evolutionary':
    print('Evolutionary Search')
    search_obj = GASearchCV(
        estimator=pipeline, param_grid=param_grid, cv=n_splits, verbose=3, n_jobs=-1,
        population_size=candidates, generations=generations, tournament_size=tournament_size, elitism=elitism
    )

orientation_cols = ['Lie on Back']

sliced_data_df = train_sample_df[train_sample_df['orientation'].isin(orientation_cols)]

Grid Search


In [7]:
y = sliced_data_df[['sequence_id', model_target]]
groups = sliced_data_df['sequence_id']

if search_mode == 'evolutionary':
    search_obj.fit(sliced_data_df, y)
else:
    search_obj.fit(sliced_data_df, y, groups=groups)


Fitting 3 folds for each of 1 candidates, totalling 3 fits


E0000 00:00:1777462845.781952      59 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777462845.790182      59 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777462845.810349      59 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777462845.810401      59 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777462845.810406      59 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777462845.810409      59 computation_placer.cc:177] computation placer already registered. Please check linka

In [8]:
# --- 1. Model Prediction & Evaluation ---
best_model = search_obj.best_estimator_
X_test = test_sample_df.copy()

# Get unique ground truth labels per sequence
y_true_seq = (test_sample_df[['sequence_id', model_target]]
              .drop_duplicates('sequence_id')
              .reset_index(drop=True))

y_pred_seq = best_model.predict(X_test)

# Calculate Accuracy
test_accuracy = accuracy_score(y_true_seq[model_target], y_pred_seq)

print(f"--- Final Test Results ---")
print(f"Best CV Score: {search_obj.best_score_:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print("\nClassification Report:\n", classification_report(y_true_seq[model_target], y_pred_seq))

# --- 2. Cleanly Append Test Results to CV Results ---
if hasattr(search_obj, 'cv_results_'):
    # Convert search results to DataFrame
    cv_results_df = pd.DataFrame(search_obj.cv_results_)
    
    # Create a "Final Test" row matching the CV columns
    # We use 'params' to label it and put the accuracy in 'mean_test_score'
    test_result_row = pd.DataFrame({
        'params': ['FINAL_HOLD_OUT_TEST'],
        'mean_test_score': [test_accuracy],
        'std_test_score': [0],
        'rank_test_score': [0]
    })
    
    # Concat results - holes in the table (like split scores) fill with NaN
    final_report_df = pd.concat([cv_results_df, test_result_row], ignore_index=True)
    
    # Save the consolidated report
    file_path = f"{model_run_folder_name}{search_mode}_lgbm_v2_results.csv"
    final_report_df.to_csv(file_path, index=False)
    
    print(f"Results consolidated and saved to: {file_path}")

--- Final Test Results ---
Best CV Score: 0.5239
Test Accuracy: 0.3997

Classification Report:
                precision    recall  f1-score   support

   pinch skin       0.39      0.30      0.34       162
    pull hair       0.55      0.40      0.46       243
pull hairline       0.39      0.15      0.21        81
      scratch       0.32      0.63      0.43       162

     accuracy                           0.40       648
    macro avg       0.41      0.37      0.36       648
 weighted avg       0.43      0.40      0.39       648

Results consolidated and saved to: model_runs/grid_lgbm_v2_results.csv
